# ISLES'26 — Exploratory Data Analysis

This notebook covers:
1. Dataset statistics (lesion volumes, days post-stroke distribution)
2. Visual inspection of sample cases (acute / sub-acute / chronic)
3. Centre-wise data distribution
4. Class imbalance analysis
5. Prediction result visualisation

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns

PROCESSED_DIR = Path('../data/processed')
SPLITS_FILE   = Path('../data/splits/splits_5fold.json')

# Load all metadata
records = []
for meta_path in sorted(PROCESSED_DIR.rglob('metadata.json')):
    with open(meta_path) as f:
        records.append(json.load(f))

df = pd.DataFrame(records)
print(df.shape)
df.head()

In [ ]:
# Days post-stroke distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['days_post_stroke'].hist(bins=50, ax=axes[0])
axes[0].set_title('Days post-stroke distribution')
df['center'].value_counts().head(30).plot.bar(ax=axes[1])
axes[1].set_title('Samples per centre (top 30)')
plt.tight_layout()
plt.show()

In [ ]:
# Compute lesion volumes
volumes = []
for _, row in df.iterrows():
    lbl_path = row.get('label_path')
    if lbl_path and Path(lbl_path).exists():
        nib_img = nib.load(lbl_path)
        vox_vol = np.prod(nib_img.header.get_zooms()[:3]) * 0.001  # mm^3 → mL
        vol = nib_img.get_fdata().sum() * vox_vol
        volumes.append(vol)

df_vol = pd.Series(volumes, name='lesion_volume_ml')
print(df_vol.describe())
df_vol.hist(bins=60)
plt.xlabel('Lesion volume (mL)')
plt.title('Lesion volume distribution')
plt.show()